# M1A3 — Atividade Prática: Construindo seu primeiro agente ReAct

Adaptação do notebook da aula com:
- Suporte **dual provider** (OpenAI **ou** Google Gemini 2.5-flash) via `.env`.
- Duas ferramentas novas pedidas pela atividade: `calcular_idade` e `converter_moeda`.
- `known_actions` atualizado.
- Perguntas de teste que forçam o ciclo **Pensamento → Ação → PAUSA → Observação → Resposta**.

## 1. Preparando dependências e variáveis de ambiente

Lê `.env`, escolhe provider (`gemini` por padrão) e instancia o cliente apropriado.

In [1]:
import os
import re
from datetime import date
from dotenv import load_dotenv

load_dotenv()

PROVIDER = os.getenv("LLM_PROVIDER", "gemini").lower()
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

if PROVIDER == "openai":
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY ausente no .env"
    from openai import OpenAI
    _openai = OpenAI()
elif PROVIDER == "gemini":
    assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY ausente no .env"
    from google import genai
    from google.genai import types as genai_types
    _gemini = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
else:
    raise ValueError(f"LLM_PROVIDER desconhecido: {PROVIDER}")

print(f"Provider ativo: {PROVIDER}")

Provider ativo: gemini


## 2. Smoke test do modelo
Uma chamada simples só pra validar credenciais e conexão antes de subir a estrutura do agente.

In [2]:
def smoke_test(texto: str) -> str:
    if PROVIDER == "openai":
        r = _openai.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0,
            messages=[{"role": "user", "content": texto}],
        )
        return r.choices[0].message.content
    r = _gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=texto,
        config=genai_types.GenerateContentConfig(temperature=0),
    )
    return r.text

print(smoke_test("Bom dia, como vai?"))

Bom dia! Eu vou bem, obrigado por perguntar. Como sou uma inteligência artificial, estou sempre pronto para ajudar.

E você, como está? Como posso te ajudar hoje?


## 3. Estrutura base do agente
Classe `Agent` que armazena o histórico e abstrai a chamada ao LLM. O método `execute()` despacha pra OpenAI ou Gemini conforme `PROVIDER`.

In [3]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        if PROVIDER == "openai":
            completion = _openai.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=0,
                messages=self.messages,
            )
            return completion.choices[0].message.content

        # Gemini: system_instruction separado, roles user/model
        system_msg = next((m["content"] for m in self.messages if m["role"] == "system"), None)
        contents = []
        for m in self.messages:
            if m["role"] == "system":
                continue
            role = "model" if m["role"] == "assistant" else "user"
            contents.append({"role": role, "parts": [{"text": m["content"]}]})
        resp = _gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=contents,
            config=genai_types.GenerateContentConfig(
                system_instruction=system_msg,
                temperature=0,
            ),
        )
        return resp.text.strip()

## 4. Prompt ReAct (system) — com as 4 ferramentas

Inclui as duas ferramentas originais (`calcular`, `preco_prato`) **e** as duas novas pedidas pela atividade (`calcular_idade`, `converter_moeda`).

In [4]:
prompt = """
Você executa em um ciclo de Pensamento, Ação, PAUSA, Observação.
No final do ciclo você fornece uma Resposta.
Use Pensamento para descrever seus pensamentos sobre a pergunta que foi feita.
Use Ação para executar uma das ações disponíveis - então retorne PAUSA.
Observação será o resultado da execução dessas ações.

Suas ações disponíveis são:

calculate:
ex: calculate: 4 * 7 / 3
Executa um cálculo e retorna o número — usa Python eval(), então use ponto para decimais.

preco_prato:
ex: preco_prato: Feijoada
Retorna o preço do prato quando fornecido o nome.

calcular_idade:
ex: calcular_idade: 1995
Recebe um ano de nascimento (4 dígitos) e retorna a idade atual da pessoa.

converter_moeda:
ex: converter_moeda: 10 USD BRL 5.00
Recebe '<valor> <moeda_origem> <moeda_destino> <taxa>' e retorna o valor convertido.
Se a taxa não for fornecida (formato '<valor> <origem> <destino>'), usa taxas internas conhecidas (USD/BRL, EUR/BRL).

Exemplo de sessão:

Pergunta: Quanto custa uma Moqueca?
Pensamento: Devo verificar o preço da Moqueca usando preco_prato.
Ação: preco_prato: Moqueca
PAUSA

Você será chamado novamente com isto:

Observação: Uma Moqueca custa R$ 89,90

Você então fornece:

Resposta: Uma Moqueca custa R$ 89,90
""".strip()

## 5. Ferramentas (funções Python) + `known_actions`

As 4 ferramentas que o agente pode acionar. Cada uma recebe **uma string** e retorna **uma string** (ou número) — o agente passa o `action_input` direto.

In [5]:
def calculate(formula: str):
    return eval(formula)

PRECOS = {
    "Feijoada": "R$ 75,90",
    "Moqueca": "R$ 89,90",
    "Picanha": "R$ 129,90",
}

def preco_prato(nome: str) -> str:
    nome = nome.strip()
    if nome in PRECOS:
        return f"Uma {nome} custa {PRECOS[nome]}"
    return f"Prato '{nome}' não encontrado no cardápio."

def calcular_idade(ano_nascimento: str) -> str:
    ano = int(ano_nascimento.strip())
    ano_atual = date.today().year
    return f"Quem nasceu em {ano} tem {ano_atual - ano} anos em {ano_atual}."

TAXAS_PADRAO = {
    ("USD", "BRL"): 5.00,
    ("BRL", "USD"): 0.20,
    ("EUR", "BRL"): 5.50,
    ("BRL", "EUR"): 0.18,
}

def converter_moeda(parametros: str) -> str:
    partes = parametros.strip().split()
    if len(partes) == 4:
        valor = float(partes[0])
        de, para = partes[1].upper(), partes[2].upper()
        taxa = float(partes[3])
    elif len(partes) == 3:
        valor = float(partes[0])
        de, para = partes[1].upper(), partes[2].upper()
        taxa = TAXAS_PADRAO.get((de, para))
        if taxa is None:
            return (
                f"Taxa {de}->{para} desconhecida. "
                f"Forneça a taxa: '<valor> <origem> <destino> <taxa>'."
            )
    else:
        return "Formato inválido. Use '<valor> <origem> <destino>' ou '<valor> <origem> <destino> <taxa>'."
    return f"{valor} {de} equivale a {valor * taxa:.2f} {para}"

known_actions = {
    "calculate": calculate,
    "preco_prato": preco_prato,
    "calcular_idade": calcular_idade,
    "converter_moeda": converter_moeda,
}

## 6. Ciclo manual (didático)

Instanciamos `campeao` e simulamos um ciclo passo a passo para ver Pensamento → Ação → PAUSA, executar a tool manualmente e devolver `Observação:` ao agente.

In [6]:
campeao = Agent(prompt)
print(campeao("Quanto custa uma Moqueca?"))

Ação: preco_prato: Moqueca
PAUSA


In [7]:
obs = preco_prato("Moqueca")
print("Observação:", obs)
print(campeao(f"Observação: {obs}"))

Observação: Uma Moqueca custa R$ 89,90
Resposta: Uma Moqueca custa R$ 89,90


## 7. Loop automático — `query()`

Detecta linhas `Ação: nome: input`, executa a ação correspondente em `known_actions`, devolve `Observação:` ao agente e repete até receber uma `Resposta:` (sem mais ações).

In [8]:
# Aceita 'Ação' (com acento, do prompt) e 'Acao' (sem, fallback caso o modelo perca o acento)
action_re = re.compile(r"^A(?:ção|cao): (\w+): (.*)$")

def query(question: str, max_turns: int = 5):
    bot = Agent(prompt)
    next_prompt = question
    for turn in range(1, max_turns + 1):
        result = bot(next_prompt)
        print(f"--- turno {turn} ---")
        print(result)
        actions = [action_re.match(line) for line in result.split("\n") if action_re.match(line)]
        if not actions:
            return result
        action, action_input = actions[0].groups()
        if action not in known_actions:
            raise Exception(f"Ação desconhecida: {action}: {action_input}")
        print(f" -> executando {action}({action_input!r})")
        observation = known_actions[action](action_input)
        print(f" <- Observação: {observation}")
        next_prompt = f"Observação: {observation}"
    return None

## 8. Testes da atividade

Perguntas que forçam o agente a usar as ferramentas **novas**.

### 8.1 — Calcular idade
Deve usar `calcular_idade: 1995`.

In [9]:
query("Quantos anos tem alguém que nasceu em 1995?")

--- turno 1 ---
Ação: calcular_idade: 1995
PAUSA
 -> executando calcular_idade('1995')
 <- Observação: Quem nasceu em 1995 tem 31 anos em 2026.
--- turno 2 ---
Resposta: Quem nasceu em 1995 tem 31 anos em 2026.


'Resposta: Quem nasceu em 1995 tem 31 anos em 2026.'

### 8.2 — Converter moeda com taxa fornecida
Deve usar `converter_moeda: 10 USD BRL 5.00`.

In [10]:
query("Quanto é 10 dólares em reais, considerando que 1 USD = 5,00 BRL?")

--- turno 1 ---
Ação: converter_moeda: 10 USD BRL 5.00
PAUSA
 -> executando converter_moeda('10 USD BRL 5.00')
 <- Observação: 10.0 USD equivale a 50.00 BRL
--- turno 2 ---
Resposta: 10 dólares equivalem a 50,00 reais.


'Resposta: 10 dólares equivalem a 50,00 reais.'

### 8.3 — Pergunta composta (idade + conversão)
Mistura as duas tools novas em uma única conversa.

In [11]:
query(
    "Uma pessoa nasceu em 1980 e quer saber a idade dela; "
    "depois converta 50 dólares em reais usando a taxa 1 USD = 5,15 BRL."
)

--- turno 1 ---
Pensamento: Devo primeiro calcular a idade da pessoa nascida em 1980 usando a ferramenta `calcular_idade`. Depois, devo converter 50 dólares para reais usando a taxa fornecida de 5,15 BRL por dólar, utilizando a ferramenta `converter_moeda`.
Ação: calcular_idade: 1980
PAUSA
 -> executando calcular_idade('1980')
 <- Observação: Quem nasceu em 1980 tem 46 anos em 2026.
--- turno 2 ---
Pensamento: A idade da pessoa já foi calculada. Agora preciso converter 50 dólares para reais usando a taxa de 5,15 BRL.
Ação: converter_moeda: 50 USD BRL 5.15
PAUSA
 -> executando converter_moeda('50 USD BRL 5.15')
 <- Observação: 50.0 USD equivale a 257.50 BRL
--- turno 3 ---
Pensamento: Já calculei a idade da pessoa e converti o valor de dólares para reais. Agora posso fornecer a resposta completa.
Resposta: Uma pessoa nascida em 1980 tem 46 anos em 2026. Além disso, 50 dólares equivalem a 257,50 reais.


'Pensamento: Já calculei a idade da pessoa e converti o valor de dólares para reais. Agora posso fornecer a resposta completa.\nResposta: Uma pessoa nascida em 1980 tem 46 anos em 2026. Além disso, 50 dólares equivalem a 257,50 reais.'